<center> <h1>Advanced RAG Concepts : Source Handling - Part 1</h1>

### 1. Packages & Models

#### 1.1 Packages

In [2]:
%pip install -U unstructured-client langchain-openai langchain-aws langchain-qdrant

Note: you may need to restart the kernel to use updated packages.


#### 1.2 Models

##### 1.2.1 Embedding Model on LM Studio

In [10]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    openai_api_base="http://localhost:1234/v1", 
    api_key="type-anything-here",
    model="text-embedding-bge-large-en-v1.5",
    check_embedding_ctx_length=False
)

##### 1.2.2 LLM on Amazon Bedrock

In [5]:
import yaml

with open('../secrets.yml', 'r') as file:
    credentials = yaml.safe_load(file)

In [4]:
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="us.meta.llama3-3-70b-instruct-v1:0",
    region="us-east-1",
    aws_access_key_id=credentials["bedrock"]["access_key"],
    aws_secret_access_key=credentials["bedrock"]["secret_key"]
)

#### 1.3 Disable Unnecessary Logging

In [31]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

### 2. Data Preparation

#### 2.1 Sample Documents

For this tutorial, our sample documents are wikipedia articles (downloaded as pdf) for the following companies :
* [BNP Paribas](https://en.wikipedia.org/wiki/BNP_Paribas)
* [STMicroelectronics](https://en.wikipedia.org/wiki/STMicroelectronics)
* [Palantir Technologies](https://en.wikipedia.org/wiki/Palantir_Technologies)

#### 2.2 OCR & Chunking

##### 2.2.1 Set up Unstructured Client

In [6]:
import unstructured_client
from unstructured_client.models import operations, shared

client = unstructured_client.UnstructuredClient(
    api_key_auth=credentials["unstructured"]["api_key"],
    server_url="https://api.unstructured.io/general/v0/general",
)

##### 2.2.2 Define Processing Function

In [7]:
import random

def process_document(filepath):
    with open(filepath, "rb") as f:
        files = shared.Files(
            content=f.read(),
            file_name=filepath
        )

    req = operations.PartitionRequest(
        partition_parameters = shared.PartitionParameters(
            files=files,
            languages=["eng"],
            chunking_strategy="by_title",
            multipage_sections=True
        )
    )

    res = client.general.partition(request=req)
    element_dicts = [element for element in res.elements]
    return element_dicts


def show_random_chunks(elements):
    view = random.sample(range(len(elements)), 3)
    for i in view:
        e = elements[i]
        print(f"Chunk N°{i+1} : {e['text']}\n\n"+20*"="+"\n\n")

##### 2.2.3 Process Sample Documents

In [32]:
bnp_elements = process_document("BNP_Paribas.pdf")
show_random_chunks(bnp_elements)

Chunk N°23 : Financial data

In 2022, total revenues of €50.4 billion represent an increase of 9% compared to 2021, BNP Paribas remains at the top of the French banks' ranking in terms of activity. During this year, BNP Paribas Group net income attributable to equity holders increased to 7.5% (to 10.2 billion euros). The geographic breakdown of Net Banking Income (NBI) at the end of



Chunk N°47 : The Maison dorée, home to Parisian operations of BNP Paribas CIB, with the registered office of BNP Paribas in the background



Chunk N°56 : dollar-dominated transactions. The fine exceeded the bank's $6.4 billion 2013 annual income and the $1.1 billion it previously had allocated for the anticipated fine.[60][61]





In [61]:
bnp_elements[0]

{'type': 'CompositeElement',
 'element_id': '8fcec7946ce4f03517e36154f4fdb916',
 'text': '% WIKIPEDIA & The Free Encyclopedia\n\nBNP Paribas',
 'metadata': {'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'orig_elements': 'eJy1UU1LxDAQ/SshoKdtaNI23XpTXKGIUlDwUJaSNtM20C/aLG5Z/O8mrQuinkSP7828mXnz0hOGBlrodKYkvkI4d33JPBY6YSSZ44sInKhkriNFKdiWU8oZ4A3CLWghhRZGc8JF349SdULDtOBGzP1BZzWoqtaGYZ7PjOaDflVS14alPOCGHXrVaatLU0o94m8QZZRE+w06Y+aGxLXYD1yy/Y7XfkPgaZ40tNZHoo7QPA2iAPxmChI0FFr1XVY0YpqyYexz02bkkeuZFbhUDeh5gEWbPODl3K46iGrxlGLoKmxXDIbJukObw2g92OEajtYlvkAv8X2c7G7ja3SJnmtAdyMA2nXFXDT9AFIJO/e8Jm7NKHvd1wxcmXNZbEsHBKeOL0PPEbKQjs8BqKQl8CD/xwxCYihGOaFrBisOOQnXnweE/oCX/t9lwH2XsT/K4OYxQYkYVS6mz79+VroxV+zfAQkc2+A=',
  'filename': 'BNP_Paribas.pdf'}}

In [33]:
stm_elements = process_document("STMicroelectronics.pdf")
show_random_chunks(stm_elements)

Chunk N°22 : ST Ericsson was a multinational manufacturer of wireless products and semiconductors, supplying to mobile device manufacturers.[8] ST-Ericsson was a 50/50 joint venture of STMicroelectronics and Ericsson established on February 3, 2009, and dissolved on August 2, 2013. Headquartered in Geneva, Switzerland, it was a fabless company, outsourcing semiconductor manufacturing to foundry companies.



Chunk N°37 : "Numonyx" in 2008. A new manufacturing facility for silicon carbide (SiC) substrates of 150 mm should open here in 2023.[23]



Chunk N°17 : On 8 December 1994, the company completed its initial public offering on the Paris and New York stock exchanges. Owner Thomson SA sold its stake in the company in 1998 when the company also listed on the Italian Bourse in Milan. In 2002, Motorola and TSMC joined ST and Philips in a new technology partnership. The Crolles 2 Alliance was created with a new 12" wafer manufacturing facility located in Crolles, France. In 2005, chief e

In [34]:
palantir_elements = process_document("Palantir_Technologies.pdf")
show_random_chunks(palantir_elements)

Chunk N°7 : Palantir Foundry is used for data integration and analysis by corporate clients such as Morgan Stanley, Merck KGaA, Airbus, Wejo, Lilium, PG&E and Fiat Chrysler Automobiles.[11] Palantir Apollo is a platform to facilitate continuous integration/continuous delivery (CI/CD) across all environments.[12][13]



Chunk N°113 : On September 28, 2020, Amnesty International released a report criticizing Palantir failure to conduct human rights due diligence around its contracts with ICE. Concerns around Palantir's rights record were being scrutinized for contributing to human rights violations of asylum-seekers and migrants.[144][145]

"HHS Protect Now" and privacy concerns (since 2020)



Chunk N°120 : Board of directors

As of December 2024, the board of directors of Palantir includes:[151]

=

Alex Karp, CEO of Palantir

[]

Alexander Moore, co-founder and former CEO of NodePrime

[]

Alexandra Schiff, former reporter of The Wall Street Journal

[]

Stephen Cohen, co-founder and 

#### 2.3 Vectorization

##### 2.3.1 Set up Qdrant client

In [35]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from langchain_core.documents import Document

qdrant_clt = QdrantClient(":memory:")

qdrant_clt.create_collection(
    collection_name="embedding_tutorial",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

qdrant = QdrantVectorStore(
    embedding=embedding,
    client=qdrant_clt,
    collection_name="embedding_tutorial",
    retrieval_mode=RetrievalMode.DENSE,
)

##### 2.3.2 Define Vectorization Function

In [36]:
def vectorize_elements(elements):
    documents = []
    ids = []
    for e in elements:
        document = Document(page_content=e["text"], metadata=e["metadata"])
        documents.append(document)
        ids.append(e["element_id"])
    qdrant.add_documents(documents=documents, ids=ids)
    print(f"Successfully upserted {len(documents)} vectors to Qdrant !")

##### 2.3.3 Vectorize Sample Documents

In [37]:
vectorize_elements(bnp_elements)

Successfully upserted 66 vectors to Qdrant !


In [38]:
vectorize_elements(stm_elements)

Successfully upserted 60 vectors to Qdrant !


In [39]:
vectorize_elements(palantir_elements)

Successfully upserted 127 vectors to Qdrant !


### 3. RAG Pipeline

#### 3.1 Retrieve

In [40]:
from langchain_core.runnables import RunnablePassthrough

retriever = qdrant.as_retriever(search_kwargs={"k": 10})

retriever_runnable = {"query": RunnablePassthrough(), "docs": retriever}

#### 3.2 Augment

In [41]:
from langchain_core.runnables import RunnableLambda

def augment_context(input_dict):
    query = input_dict["query"]
    docs = input_dict["docs"]
    context = ""
    sourcing = {}
    for i, doc in enumerate(docs):
        sourcing[i+1] = {"text": doc.page_content, **doc.metadata}
        context += f"Snippet {i+1}: {doc.page_content}" + "<end of snippet>\n"
    return {"query": query, "context": context, "sourcing": sourcing}

augmenter_runnable = RunnableLambda(augment_context)

#### 3.3 Generate

In [42]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from textwrap import dedent

system_prompt = """
You are a data-based question answering assistant. 
You only base your answers on the data provided to you along with the question.
You always refer to the specific origin of the information you used for each part of your answer.
You are given a question and a set of snippets of texts to help you answer the question.
The snippets are provided to you in the following format: 'Snippet <number>: <text>'. A '<end of snippet>' tag signals the end of a snippet.
If you use information from a snippet to make a statement, your statement (the answer to the question) should be followed by the snippet number in this format : **(<number>)**.
For example, if you use 'Snippet 4' to answer a question, your answer should look like this : '<answer> **(4)**'.
If you combined multiple snippets (e.g. 5 and 3) to answer a question, your answer should look like this : '<first statement> **(5)**, <second statement> **(3)**, ...'.
If you don't have enough information to answer, just say the following failure message : 'There wasn't enough information to answer the question.'.
"""

user_prompt = """
Question: {query}
Context: {context}
Answer: 
"""

prompt = ChatPromptTemplate([
    ("system", dedent(system_prompt)),
    ("user", dedent(user_prompt)),
])

generator_runnable = prompt | llm | StrOutputParser()

In [43]:
chain = retriever_runnable | augmenter_runnable | generator_runnable

In [44]:
result = chain.invoke("When was BNP Paribas founded ?")
print(result)

BNP Paribas was founded in 2000 from the merger of two of France's foremost financial institutions, Banque Nationale de Paris (BNP) and Paribas **(2)**, with the merger taking place on 23 May 2000 **(1)**, **(3)**, **(5)**.


#### 3.4 Source

##### 3.4.1 Prettify Sourced Answer

In [45]:
import re

superscripts = "⁰¹²³⁴⁵⁶⁷⁸⁹"

def int_to_superscript(integer):
    return ''.join(superscripts[int(d)] for d in str(integer))

def superscript_to_int(superscript):
    return ''.join(str([c for c in superscripts].index(d)) for d in superscript)

def prettify_sources(text):
    pattern = r"\*\*\((\d+)\)\*\*"
    matches = re.findall(pattern, text)
    integers = list(map(int, matches))
    superscript_map = {}
    seen = set()
    i = 0
    for integer in integers:
        if integer not in seen:
            superscript_map[integer] = int_to_superscript(i + 1)
            i += 1
            seen.add(integer)
    replaced_text = text
    for i, old_value in enumerate(matches):
        new_value = superscript_map[int(old_value)]
        replaced_text = replaced_text.replace(f" **({old_value})**", new_value, 1)
    
    superscript_to_old = {v: k for k, v in superscript_map.items()}
    
    return replaced_text, superscript_to_old

##### 3.4.2 Sourced RAG Chain

In [47]:
def show_source(input_dict):
    answer = input_dict["answer"]
    sourcing = input_dict["sourcing"]
    new_answer, source_matching = prettify_sources(answer)
    new_answer += "\n\nSources:\n"
    for superscript, integer in source_matching.items():
        src_meta = sourcing[integer]
        new_answer += superscript + src_meta["filename"] + f", page n°{src_meta['page_number']}.\n"
    return new_answer.strip()

source_runnable = RunnableLambda(show_source)

In [48]:
srag_chain = retriever_runnable | augmenter_runnable | RunnablePassthrough.assign(answer=generator_runnable) | source_runnable

##### 3.4.3 Tests/Examples

In [51]:
result = srag_chain.invoke("When and how was STMicroelectronics founded ?")

In [52]:
print(result)

STMicroelectronics was founded in 1987¹ from the merger of two state-owned semiconductor corporations: Thomson Semiconducteurs of France and SGS Microelettronica of Italy¹, which were two government-owned semiconductor companies². At the time of the merger, the new corporation was named SGS-THOMSON³ and was led by chief executive officer Pasquale Pistorio³. The company took its current name of STMicroelectronics in May 1998³.

Sources:
¹STMicroelectronics.pdf, page n°1.
²STMicroelectronics.pdf, page n°1.
³STMicroelectronics.pdf, page n°2.


In [59]:
result = srag_chain.invoke("What are the major products of Palantir Technologies ?")

In [60]:
print(result)

The major products of Palantir Technologies are Palantir Gotham, Palantir Foundry, Palantir Apollo, and Palantir AIP¹. Additionally, the company also offers Palantir Metropolis², which is used by hedge funds, banks, and financial services firms³. Palantir Foundry is used for data integration and analysis⁴, Palantir Gotham is an intelligence and defense tool¹, and Palantir Apollo is a platform to facilitate continuous integration/continuous delivery (CI/CD)⁴.

Sources:
¹Palantir_Technologies.pdf, page n°2.
²Palantir_Technologies.pdf, page n°7.
³Palantir_Technologies.pdf, page n°7.
⁴Palantir_Technologies.pdf, page n°2.
